# M2-T05 Brain MRI Viewer — Colab
This notebook uses the existing project cleaning pipeline. It does not download the dataset to your PC.

**Flow:** Google Drive archive → extract in Colab → M1-T04 standardize → `/content/processed` → Streamlit viewer.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted


In [ ]:
from pathlib import Path
import os, tarfile, zipfile, shutil, glob

PROJECT = Path("/content/drive/MyDrive/BrainMRI_Data")
# Change this only if your repo is somewhere else in Colab.
print("PROJECT:", PROJECT)

print("Top Drive folders:")
for p in sorted(Path("/content/drive/MyDrive/BrainMRI_Data").glob("*"))[:30]:
    print(" -", p)

PROJECT: /content/drive/MyDrive/BrainMRI_Data
Top Drive folders:
 - /content/drive/MyDrive/BrainMRI_Data/Reports
 - /content/drive/MyDrive/BrainMRI_Data/apps
 - /content/drive/MyDrive/BrainMRI_Data/manifests
 - /content/drive/MyDrive/BrainMRI_Data/mlflow.db
 - /content/drive/MyDrive/BrainMRI_Data/mlruns
 - /content/drive/MyDrive/BrainMRI_Data/pipeline
 - /content/drive/MyDrive/BrainMRI_Data/processed
 - /content/drive/MyDrive/BrainMRI_Data/qc
 - /content/drive/MyDrive/BrainMRI_Data/raw
 - /content/drive/MyDrive/BrainMRI_Data/runs
 - /content/drive/MyDrive/BrainMRI_Data/src


## 1) Put the project in Colab
Upload/clone the project so this path exists:
`/content/NHA-5-187-main`

If you already have the repo elsewhere, change `PROJECT` in the next cell.

In [ ]:
if not PROJECT.exists():
    raise FileNotFoundError(f"Project not found: {PROJECT}. Put the repository there or change PROJECT.")
print("Project found")
print("standardize.py:", (PROJECT/"pipeline/standardize.py").exists())
print("viewer:", (PROJECT/"apps/m2_t05_viewer.py").exists())

Project found
standardize.py: True
viewer: True


In [ ]:
# Search Drive for the archive without copying it locally.
patterns = ["**/*.tar", "**/*.tar.gz", "**/*.tgz", "**/*.zip"]
found=[]
root=Path("/content/drive/MyDrive/BrainMRI_Data")
for pat in patterns:
    for p in root.glob(pat):
        n=p.name.lower()
        if "brats" in n or "brain" in n or "training" in n:
            found.append(p)
for p in sorted(set(found))[:50]:
    print(p)
if not found:
    print("No obvious BraTS archive found. Set ARCHIVE manually in the next cell.")

/content/drive/MyDrive/BrainMRI_Data/processed/brats2021_trainval_v1.tar
/content/drive/MyDrive/BrainMRI_Data/raw/brats2021/BraTS2021_00495.tar
/content/drive/MyDrive/BrainMRI_Data/raw/brats2021/BraTS2021_00621.tar
/content/drive/MyDrive/BrainMRI_Data/raw/brats2021/BraTS2021_Training_Data.tar
/content/drive/MyDrive/BrainMRI_Data/raw/external_brats_africa/external_brats_africa.tar


In [ ]:
# SET THIS only if auto-discovery did not find your archive.
ARCHIVE = None  # e.g. Path("/content/drive/MyDrive/BraTS2021_Training_Data.tar")

if ARCHIVE is None and found:
    ARCHIVE = found[0]
ARCHIVE = Path(ARCHIVE) if ARCHIVE else None
print("ARCHIVE:", ARCHIVE)

ARCHIVE: /content/drive/MyDrive/BrainMRI_Data/processed/brats2021_trainval_v1.tar


## 2) Extract one case (fast path)
The viewer only needs one complete case. We inspect the archive and extract the first case containing all four MRI modalities plus segmentation.

In [ ]:
RAW = Path("/content/drive/MyDrive/BrainMRI_Data/raw")
PROC = Path("/content/drive/MyDrive/BrainMRI_Data/processed")
RAW.mkdir(exist_ok=True)
PROC.mkdir(exist_ok=True)

if ARCHIVE is None or not ARCHIVE.exists():
    raise FileNotFoundError("Set ARCHIVE to the archive file inside Google Drive.")

required={"t1","t1ce","t2","flair","seg"}

def case_key(name):
    parts=Path(name).parts
    # Find a directory that contains modality files.
    return parts[:-1]

member_names=[]
if ARCHIVE.suffixes[-2:] == [".tar", ".gz"] or ARCHIVE.suffix==".tar":
    with tarfile.open(ARCHIVE, "r:*") as tf:
        member_names=[m.name for m in tf.getmembers() if m.isfile() and m.name.endswith(".nii.gz")]
else:
    with zipfile.ZipFile(ARCHIVE) as z:
        member_names=[n for n in z.namelist() if n.endswith(".nii.gz")]

print("NIfTI files in archive:", len(member_names))

# Group files by parent directory and find the first complete case.
candidates={}
for n in member_names:
    base=Path(n).name
    if base.endswith(".nii.gz"):
        stem=base[:-7]
        if stem in required:
            candidates.setdefault(str(Path(n).parent), set()).add(stem)
complete=[p for p,s in candidates.items() if required.issubset(s)]
print("Complete cases found in archive:", len(complete))
if not complete:
    raise RuntimeError("No complete case with t1/t1ce/t2/flair/seg was found. The archive layout may differ from the expected BraTS layout.")
case_parent=complete[0]
print("Extracting:", case_parent)

files_to_extract=[]
for n in member_names:
    if str(Path(n).parent)==case_parent:
        files_to_extract.append(n)

if ARCHIVE.suffixes[-2:] == [".tar", ".gz"] or ARCHIVE.suffix==".tar":
    with tarfile.open(ARCHIVE,"r:*") as tf:
        for n in files_to_extract:
            m=tf.getmember(n)
            tf.extract(m, path=RAW)
else:
    with zipfile.ZipFile(ARCHIVE) as z:
        for n in files_to_extract:
            z.extract(n, path=RAW)

for n in files_to_extract:
    print(n)

NIfTI files in archive: 5315
Complete cases found in archive: 1063
Extracting: brats2021/BraTS2021_01110/0


/tmp/ipykernel_1335/825611036.py:50: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extract(m, path=RAW)


brats2021/BraTS2021_01110/0/t1.nii.gz
brats2021/BraTS2021_01110/0/flair.nii.gz
brats2021/BraTS2021_01110/0/t1ce.nii.gz
brats2021/BraTS2021_01110/0/seg.nii.gz
brats2021/BraTS2021_01110/0/t2.nii.gz


## 3) Run the project standardization (M1-T04)
This produces the exact cleaned layout expected by M2-T05.

In [70]:
import sys
sys.path.insert(0, str(PROJECT/"src"))
from bmts.data.cleaning.standardize import regenerate as rg

# Find the extracted case directory.
ni=sorted(RAW.rglob("*.nii.gz"))
parents={}
for p in ni:
    parents.setdefault(p.parent,[]).append(p.name)
case_dir=max(parents, key=lambda p: len(parents[p]))
print("Raw case:", case_dir)
print("Files:", sorted(parents[case_dir]))

# The standardize module in this project may expect an explicit mapping.
import inspect
print(inspect.signature(rg))


Raw case: /content/drive/MyDrive/BrainMRI_Data/raw/brats2021/BraTS2021_01110/0
Files: ['flair.nii.gz', 'seg.nii.gz', 't1.nii.gz', 't1ce.nii.gz', 't2.nii.gz']
(splits=('train', 'val'), datasets=('brats2021', 'mu', 'rhuh'))


In [ ]:
# Try the project function using keyword arguments inferred from its signature.
sig=inspect.signature(rg)
kwargs={}
for name,param in sig.parameters.items():
    lname=name.lower()
    if "input" in lname or lname in {"src","source","case_dir","raw_dir","input_dir"}: kwargs[name]=case_dir
    elif "output" in lname or lname in {"dst","dest","destination","processed_dir","output_dir"}: kwargs[name]=PROC/"brats2021"/case_dir.name/"0"
    elif "subject" in lname: kwargs[name]=case_dir.name
    elif param.default is inspect._empty:
        print("Unhandled required arg:", name, param.default)
print("Calling with:", kwargs)
try:
    result=rg(**kwargs)
    print("standardize_case returned:", result)
except TypeError as e:
    print("The project function signature differs from this wrapper. Error:", e)
    raise


Calling with: {}


brats2021:   0%|          | 0.00/13.4G [00:00<?, ?B/s]

mu:   0%|          | 0.00/11.9G [00:00<?, ?B/s]

rhuh:   0%|          | 0.00/3.10G [00:00<?, ?B/s]

  0%|          | 0/8253 [00:00<?, ?it/s]

status
done    8253
standardize_case returned:         dataset          subject timepoint   scan  \
0     brats2021  BraTS2021_00000         0     t1   
1     brats2021  BraTS2021_00000         0   t1ce   
2     brats2021  BraTS2021_00000         0     t2   
3     brats2021  BraTS2021_00000         0  flair   
4     brats2021  BraTS2021_00000         0    seg   
...         ...              ...       ...    ...   
9828       rhuh        RHUH-0002       2.0   t1ce   
9829       rhuh        RHUH-0002       2.0     t2   
9830       rhuh        RHUH-0002       2.0  flair   
9831       rhuh        RHUH-0002       2.0    seg   
9832       rhuh        RHUH-0002       2.0     t1   

                                                   path  \
0     /content/data/brats2021/BraTS2021_00000/BraTS2...   
1     /content/data/brats2021/BraTS2021_00000/BraTS2...   
2     /content/data/brats2021/BraTS2021_00000/BraTS2...   
3     /content/data/brats2021/BraTS2021_00000/BraTS2...   
4     /content/data/b

In [76]:
# Verify the viewer input layout.
PROC = Path("/content/drive/MyDrive/BrainMRI_Data/processed")
complete=list(PROC.rglob("seg.nii.gz"))
print("seg files:", complete[:10])
if not complete:
    raise RuntimeError("M1-T04 did not create seg.nii.gz under /content/drive/MyDrive/BrainMRI_Data/processed")
seg=complete[0]
case=seg.parent
print("Viewer case directory:", case)
print("Files:", sorted(p.name for p in case.iterdir()))

seg files: [PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0067/Timepoint_2/seg.nii.gz'), PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0067/Timepoint_1/seg.nii.gz'), PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0019/Timepoint_5/seg.nii.gz'), PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0019/Timepoint_6/seg.nii.gz'), PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0019/Timepoint_4/seg.nii.gz'), PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0168/Timepoint_1/seg.nii.gz'), PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0158/Timepoint_1/seg.nii.gz'), PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0260/Timepoint_2/seg.nii.gz'), PosixPath('/content/drive/MyDrive/BrainMRI_Data/processed/processed/mu/PatientID_0260/Timepo

## 4) Launch M2-T05
The simplest Colab option is to use Streamlit with a tunnel.

In [ ]:
%pip -q install streamlit nibabel matplotlib pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 87.3 MB/s eta 0:00:00


In [77]:
import subprocess, time, os, signal
os.environ["BMTS_PROCESSED_ROOT"]="/content/drive/MyDrive/BrainMRI_Data/processed"

# Start Streamlit in the background.
proc=subprocess.Popen(["streamlit","run",str(PROJECT/"apps/m2_t05_viewer.py"),"--server.port","8501","--server.headless","true"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(6)
print("Streamlit started on port 8501. PID:", proc.pid)


Streamlit started on port 8501. PID: 59086


In [ ]:
from pyngrok import ngrok
!ngrok config add-authtoken 3KPvSHuR0RenSe3XgG2E9aQ5tSs_7pEieoLZv6EdLqbPY8oU9
public_url = ngrok.connect(8501).public_url
print("OPEN THIS URL:", public_url)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
OPEN THIS URL: https://varsity-endurable-snitch.ngrok-free.dev
